# SiteSafe — Fine-tune Gemma 4 E4B on a free Kaggle T4 GPU

**Goal:** Reproduce the SiteSafe fine-tuned model end-to-end on Kaggle's free T4 (16 GB VRAM) accelerator. The notebook installs Unsloth, downloads the Gemma 4 E4B checkpoint, fine-tunes a LoRA adapter on the SiteSafe SFT dataset, and exports a Q4_K_M GGUF that can be loaded directly into Ollama.

**Hardware:** Kaggle → Settings → Accelerator → **GPU T4 x1**.

**Time:** ~25 min for 200 training steps at LoRA rank 16.

**Output:** `/kaggle/working/sitesafe-gemma4-e4b/sitesafe-gemma4-e4b.Q4_K_M.gguf` — download this and use with `training/Modelfile`.

---

## 1. Install Unsloth + dependencies

Unsloth provides 2× faster training and ~70 % less VRAM than vanilla HuggingFace + LoRA. Day-zero support for Gemma 4.

In [ ]:
%%capture
!pip install --upgrade pip
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets pillow

## 2. Load Gemma 4 E4B with FastVisionModel

We use 4-bit QLoRA so the 4B-parameter model fits comfortably on T4.

In [ ]:
from unsloth import FastVisionModel
import torch

MAX_SEQ_LENGTH = 2048
MODEL_NAME = 'unsloth/gemma-4-E4B-it'

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    use_gradient_checkpointing='unsloth',
)
print('Model loaded. CUDA available:', torch.cuda.is_available())

## 3. Apply LoRA adapters

If the next cell hits an OOM, set `finetune_vision_layers=False` and re-run — that fine-tunes only the language adapter, which still lifts SiteSafe quality significantly while halving memory usage.

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,    # ← set False if you OOM on T4
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    bias='none',
    target_modules='all-linear',
)
model.print_trainable_parameters()

## 4. Load the SiteSafe SFT dataset

Upload `data/training/train.jsonl` as a Kaggle Dataset (or generate it on-Kaggle by running `data/build_training_data.py`).

Each row has the shape:

```json
{
  "messages": [
    {"role": "user",      "content": [{"type": "image", "image": "..."}, {"type": "text", "text": "..."}]},
    {"role": "assistant", "content": "## SiteSafe Violation Report ..."}
  ]
}
```

In [ ]:
from datasets import Dataset
import json
from pathlib import Path

TRAIN_JSONL = Path('/kaggle/input/sitesafe-train/train.jsonl')
if not TRAIN_JSONL.exists():
    TRAIN_JSONL = Path('/kaggle/working/train.jsonl')
    raise SystemExit(
        'Upload your SiteSafe SFT JSONL as a Kaggle dataset (or copy it to '
        f'{TRAIN_JSONL}). See data/build_training_data.py in the repo.'
    )

rows = [json.loads(l) for l in TRAIN_JSONL.read_text().splitlines() if l.strip()]
print(f'Loaded {len(rows)} examples')
dataset = Dataset.from_list(rows)
dataset

## 5. Train

200 steps × effective batch size 8 (per_device 1 × grad_accum 8) ≈ 1,600 examples seen. Bump `max_steps` to 400–600 if your dataset is large enough to support it without overfitting.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=dataset,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        max_steps=200,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        output_dir='/kaggle/working/sitesafe-checkpoints',
        seed=42,
        report_to='none',
    ),
)
trainer.train()

## 6. Save merged model + export GGUF for Ollama

In [ ]:
MERGED_DIR = '/kaggle/working/sitesafe-merged'
model.save_pretrained_merged(MERGED_DIR, tokenizer)
print('Merged checkpoint:', MERGED_DIR)

In [ ]:
model.save_pretrained_gguf(
    '/kaggle/working/sitesafe-gemma4-e4b',
    tokenizer,
    quantization_method='q4_k_m',
)
!ls -lh /kaggle/working/sitesafe-gemma4-e4b/

## 7. Download + register with Ollama (locally)

1. Click **Output** in the right-hand Kaggle panel and download the `.gguf` file.
2. Drop it next to `training/Modelfile` in your local clone of the SiteSafe repo.
3. Register it with Ollama:

```bash
ollama create sitesafe -f training/Modelfile
ollama run sitesafe
```

4. Launch the SiteSafe app:

```bash
python app/sitesafe_app.py
```

Open <http://localhost:7860> and upload a construction-site photo.

**Done — you have an OSHA inspector running entirely on your laptop.**